# Round 11 | Features and matched comparison

Each arm adds 24 features to the exact saved 134-feature control. No changed candidate pool, ensemble, or ranker hyperparameter search. Run cells individually.

In [ ]:
from pathlib import Path
import json, sys, importlib.util
ROOT = Path.home() / "otto_feature_round11"
assert ROOT.is_dir(), ROOT
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("otto_launcher_11", ROOT / "launch.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
def stage(name):
    return launcher.run_stage(name)
def require(relative, key, expected):
    value = json.loads((ROOT / relative).read_text())
    assert value[key] == expected, (relative, value.get(key))
    print(expected)
    return value
print("KERNEL_READY")

## 1. Representation gate

In [ ]:
require("outputs/representations/manifest.json", "status", "ROUND11_REPRESENTATIONS_READY")

## 2. Build both feature matrices

In [ ]:
stage('features')

## 3. Require complete features; replay controls and fit challengers
The screen replays six controls before fitting at most twelve challengers. A failure is a stop, not an invitation to keep retrying.

In [ ]:
require("outputs/feature_manifest.json", "status", "ROUND11_FEATURES_READY")
stage("screen")

## 4. Recompute report from saved integer statistics

In [ ]:
stage('report')

## 5. Read the gate decision

In [ ]:
require("outputs/report_receipt.json", "status", "ROUND11_REPORT_READY")
result = json.loads((ROOT / "outputs/result.json").read_text())
print(json.dumps(result["arms"], indent=2))
print(json.dumps(result["comparisons"]["idf_session_latent_minus_control_shared"], indent=2))
print("Save this notebook, then open 03_saved_results.ipynb.")